# Pipeline Evaluation

Reads articles from `benchmark.csv`, runs them through the pipeline in-memory
(no DB writes), and compares decisions against ground-truth `should_keep` labels.

Run after any change to thresholds, BM25 terms, weights, or the LLM prompt.

In [2]:
import sys
import uuid
from datetime import datetime
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from dotenv import load_dotenv

load_dotenv()

sys.path.insert(0, '.')

from app import NewsTriageService
from models import NewsEntry

BENCHMARK_PATH = 'benchmark.csv'
EVAL_DIR       = Path('eval_runs')
EVAL_DIR.mkdir(exist_ok=True)

# Label this run — change to describe what you changed
RUN_LABEL = 'baseline'  # e.g. 'add_bm25_terms', 'lower_discard_threshold'

In [3]:
# Load benchmark ground truth
benchmark = pd.read_csv(BENCHMARK_PATH)
benchmark['should_keep'] = benchmark['should_keep'].astype(bool)
print(f'Benchmark articles: {len(benchmark)}')
print(benchmark.groupby('source')['should_keep'].agg(['sum', 'count', 'mean']).round(2))

Benchmark articles: 113
              sum  count  mean
source                        
ars-technica   28     45  0.62
reddit          4     68  0.06


In [4]:
# Build NewsEntry objects from benchmark
articles = [
    NewsEntry(
        id=row['article_id'],
        source=row['source'],
        title=row['title'],
        body=row['body'] if pd.notna(row['body']) else '',
        published_at=row['published_at'],
    )
    for _, row in benchmark.iterrows()
]
print(f'Articles ready: {len(articles)}')

Articles ready: 113


In [5]:
# Run pipeline in-memory — no DB writes
service = NewsTriageService()
run_id  = str(uuid.uuid4())

print('Running pipeline (LLM calls will take a few minutes)...')
results = service.process_articles(articles, run_id=run_id, persist=False)
print(f'Done. Kept: {len(results)}/{len(articles)}')

Running pipeline (LLM calls will take a few minutes)...
Auto-discarding article '5 plead guilty to laptop farm and ID theft scheme to land North Koreans US IT jobs' with fused score 0.306
Sending article 'Critics scoff after Microsoft warns AI feature can infect machines and pilfer data' for LLM judgment with fused score 0.483
LLM judgment completed in 4.37 seconds
Auto-discarding article 'Oops. Cryptographers cancel election results after losing decryption key.' with fused score 0.312
Auto-discarding article 'Researchers question Anthropic claim that AI-assisted attack was 90% autonomous' with fused score 0.313
Auto-discarding article 'This hacker conference installed a literal antivirus monitoring system' with fused score 0.326
Auto-discarding article 'How to know if your Asus router is one of thousands hacked by China-state hackers' with fused score 0.330
Sending article 'Admins and defenders gird themselves against maximum-severity server vuln' for LLM judgment with fused score 0.5

In [ ]:
# Build results dataframe and join with ground truth
df_results = pd.DataFrame([
    {
        'article_id':        r.id,
        'pipeline_keep':     r.keep,
        'decision_source':   r.decision_source,
        'fused_score':       r.fused_score,
        'lexical_score':     r.lexical_score,
        'semantic_score':    r.semantic_score,
        'freshness_score':   r.freshness_score,
        'llm_relevance_score': r.llm_relevance_score,
        'llm_reason':        r.llm_reason,
    }
    for r in service._last_all_results  # see note below
])

df = benchmark[['article_id', 'source', 'title', 'should_keep', 'confidence']].merge(
    df_results, on='article_id', how='inner'
)
print(f'Matched: {len(df)}')
print(df['decision_source'].value_counts())

In [ ]:
# Overall metrics
y_true = df['should_keep'].astype(int)
y_pred = df['pipeline_keep'].astype(int)

precision = precision_score(y_true, y_pred, zero_division=0)
recall    = recall_score(y_true, y_pred, zero_division=0)
f1        = f1_score(y_true, y_pred, zero_division=0)
cm        = confusion_matrix(y_true, y_pred)

print('=== Pipeline Evaluation ===')
print(f'Precision : {precision:.3f}  (of articles kept, how many should be kept)')
print(f'Recall    : {recall:.3f}  (of articles that should be kept, how many were kept)')
print(f'F1        : {f1:.3f}')
print()
print(pd.DataFrame(cm,
    index  =['GT: discard', 'GT: keep'],
    columns=['Pred: discard', 'Pred: keep']))

In [ ]:
# Metrics by source
for src, grp in df.groupby('source'):
    yt = grp['should_keep'].astype(int)
    yp = grp['pipeline_keep'].astype(int)
    print(f"{src:20}  precision={precision_score(yt,yp,zero_division=0):.2f}  "
          f"recall={recall_score(yt,yp,zero_division=0):.2f}  "
          f"f1={f1_score(yt,yp,zero_division=0):.2f}  n={len(grp)}")

In [ ]:
# False negatives — should keep, pipeline discarded
fn = df[(df['should_keep']) & (~df['pipeline_keep'])].sort_values('fused_score', ascending=False)
print(f'False negatives: {len(fn)}')
pd.set_option('display.max_colwidth', 80)
fn[['source', 'decision_source', 'fused_score', 'lexical_score',
    'semantic_score', 'freshness_score', 'confidence', 'title']].round(2)

In [ ]:
# False positives — should discard, pipeline kept
fp = df[(~df['should_keep']) & (df['pipeline_keep'])].sort_values('fused_score', ascending=False)
print(f'False positives: {len(fp)}')
fp[['source', 'decision_source', 'fused_score', 'lexical_score',
    'semantic_score', 'freshness_score', 'confidence', 'title']].round(2)

In [ ]:
# Score distributions for errors
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, data, label, color in [
    (axes[0], fn, 'False Negatives', '#dc2626'),
    (axes[1], fp, 'False Positives', '#f59e0b'),
]:
    ax.hist(data['fused_score'], bins=15, color=color, alpha=0.8, range=(0, 1))
    ax.axvline(0.45, color='black', linestyle='--', linewidth=1, label='discard threshold (0.45)')
    ax.axvline(0.75, color='black', linestyle=':',  linewidth=1, label='keep threshold (0.75)')
    ax.set_title(f'{label} — fused score distribution')
    ax.set_xlabel('Fused Score')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Save this run to eval_runs/
ts       = datetime.now().strftime('%Y%m%d_%H%M%S')
out_path = EVAL_DIR / f'{ts}_{RUN_LABEL}.csv'
df['run_label'] = RUN_LABEL
df['run_ts']    = ts
df.to_csv(out_path, index=False)
print(f'Saved: {out_path}')

# Summary row for easy comparison across runs
summary = {
    'run_label': RUN_LABEL,
    'run_ts':    ts,
    'precision': round(precision, 3),
    'recall':    round(recall, 3),
    'f1':        round(f1, 3),
    'false_negatives': len(fn),
    'false_positives': len(fp),
    'total': len(df),
}
summary_path = EVAL_DIR / 'summary.csv'
summary_df   = pd.DataFrame([summary])
if summary_path.exists():
    summary_df = pd.concat([pd.read_csv(summary_path), summary_df], ignore_index=True)
summary_df.to_csv(summary_path, index=False)
print()
print('All runs so far:')
print(summary_df.to_string(index=False))